## Visualization 1

In [2]:
## Scientific calculus
import pandas as pd
import numpy as np
from scipy.stats import gaussian_kde

## Plots
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import plotly.graph_objects as go

## Text, OS and images
from adjustText import adjust_text 
import os
from PIL import Image

### Import Dataset and recover DFs

In [3]:
## Loading the dataset from local folder
POKE_df = pd.read_csv("../DATASETS/pokemon_complete_2025.csv")

SPRITE_FOLDER = f"../POKESPRITES/"

## List of the 2020 most appreciated pokemons from the source pokemon.com
most_appreciated_pokemons2020 = [
    "greninja", "lucario", "mimikyu-disguised", "charizard",
    "umbreon", "sylveon", "garchomp", "rayquaza", "gardevoir", "gengar"
]

## Received votes

most_appreciated_pokemons2020_num = [
    140559, 102259, 99077, 93968, 67062, 66029, 61877, 60939, 60596, 60214    
]


## Since the dataset has roman nums, we need to have a map to convert
roman_to_int = {
    'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5,
    'VI': 6, 'VII': 7, 'VIII': 8, 'IX': 9
}

## FONTS and sizes params
poke_font = "Pokemon Emerald, Arial"
font_size_title = 30
font_size_text = 20

UP_TO_GEN = 8 # we consider pokemons up to gen 8, due to 2020 reference year

POKE_df['generation_num'] = POKE_df['generation'].map(roman_to_int) ## Transform roman numbers into int
POKE_df = POKE_df.dropna(subset=['generation_num']) # Drop NAs
POKE_df = POKE_df[POKE_df['generation_num'] <= UP_TO_GEN] # Limit/filter to the selected gen
POKE_df['name_lower'] = POKE_df['name'].str.lower() # Make the name lowercase (we need to match with most appr. list)
POKE_df['is_favorite'] = POKE_df['name_lower'].isin(most_appreciated_pokemons2020) # New column, to see if the pokémon is in the appr. list
POKE_df['All Pokémon'] = ''

## Create the dataset about favourites pokemons with votes used for the plot
favorites_df = POKE_df[POKE_df['is_favorite']].copy() # Create a copy of the original
votes_mapping = dict(zip(most_appreciated_pokemons2020, most_appreciated_pokemons2020_num)) # Create a (pokemon, votes)
favorites_df['votes'] = favorites_df['name_lower'].map(votes_mapping) # Add that
favorites_df_sorted = favorites_df.sort_values(by='votes', ascending=False) # Sort by votes 


### Perform KDE estimation

In [4]:
MULTIPLY_BY = 31000000
BREAKS = 500

kde = gaussian_kde(POKE_df['base_stat_total'],bw_method=0.15)
x_range = np.linspace(POKE_df['base_stat_total'].min() - 50, POKE_df['base_stat_total'].max() + 20, BREAKS)
y_density = kde.evaluate(x_range) * MULTIPLY_BY

### Building the figure

In [10]:
fig = go.Figure()


## ........................ ##
## UNDERLYING DENSITY (KDE) ##
## ........................ ##

fig.add_trace(go.Scatter(
    x=x_range,
    y=y_density,
    mode='lines',
    line=dict(color='rgba(211, 211, 211, 0.5)', width=2, shape='spline'),
    fill='tozeroy', fillcolor='rgba(211, 211, 211, 0.1)',
    hoverinfo='skip', showlegend=False,
))

## ZERO AXIS (Horizontal line at y=0)
fig.add_hline(y=0, line_color="#555555", opacity=0.5)

## ....................... ##
## AREAS AND tier SECTIONS ##
## ....................... ##

fig.add_annotation(x=250, y=205000, 
                   text="<b>Untiered (<400)</b>", showarrow=False, 
                   font=dict(family=poke_font, size=20,color="gray"))

## Add the colored rectangle 
fig.add_vrect(x0=400, x1=600, fillcolor="blue", opacity=0.06, layer="below",line_width=0)
## Ad the text annotation
fig.add_annotation(x=500, y=205000,
                   text="Standard Fully Evolved Tier", showarrow=False, 
                   font=dict(family=poke_font, size=20,color="steelblue"))

## Add the colored rectangle 
fig.add_vrect(x0=600, x1=800, fillcolor="gold", opacity=0.06, layer="below",line_width=0)
## Add the text annotation
fig.add_annotation(x=700, y=205000, text="Legendary Tier (>600)", showarrow=False, 
                   font=dict(family=poke_font, size=20,color="goldenrod"))

fig.update_layout(
    ## TITLE INFOS and general values
    title_text="<b>FAN  FAVORITES  POKéMON  DON'T  RELY  SOLELY  ON  THEIR  (Extreme)  POWER</b><br><sup>Top  10  most  liked  Pokémon  up  to  Gen  8  span  across  middle-to-high  base-stat  distribution</sup>",
    title_font=dict(family=poke_font, size=font_size_title, color = "#5A5665"),
    title_pad=dict(b=35), # Adds space below the title
    margin=dict(l=40, r=240, t=150, b=200),
    height=820, 
    plot_bgcolor="white", 
    paper_bgcolor="rgba(0,0,0,0)",
    title_x = 0.07,

    ## XAXIS 
    xaxis_title="<b>BASE STAT TOTAL</b>",
    xaxis_tickfont=dict(family=poke_font, size=20, color="#5A5665"),
    xaxis=dict(
        title_font=dict(family=poke_font, size=font_size_text, color = "#D64848"),
        tickfont=dict(family=poke_font, size=20, color="#5A5665"),
        showgrid=False,  
        zeroline=False,
        range=[100, 800],
    ),

    ## YAXIS
    yaxis_title="<b>RECEIVED VOTES</b>",
    yaxis_tickfont=dict(family=poke_font, size=20, color="#5A5665"),
    yaxis=dict(
        visible=True, 
        title_font=dict(family=poke_font, size=font_size_text, color = "#D64848"),
        range=[0, 220000],
        ticks="outside",
        showgrid=False,  
        ticklen=10,
        color = "#5A5665"
    ),
)

## .................. ##
## OVERLAY OF SPRITES ##
## .................. ##
# For each favorite pokemon, draw the vertical line to match with the votes and base stat total
for _, row in favorites_df.iterrows():
    fig.add_trace(go.Scatter(
        x=[row['base_stat_total'], row['base_stat_total']], 
        y=[0, row['votes']],
        mode='lines',
        line=dict(color='#e3350d', width=2, dash='dot'),
        showlegend=False,
        hoverinfo='skip',
        zorder=1
    ))

# For each favorite pokemon, overimpose also the sprites from local folders
for _, row in favorites_df.iterrows():
    sprite_path = SPRITE_FOLDER + f"{row['name_lower']}.png"
    fig.add_layout_image(
        dict(
            source=Image.open(sprite_path),
            xref="x", yref="y",
            x=row['base_stat_total'], 
            y=row['votes'],          
            sizex=28000,
            sizey=28000,           
            xanchor="center", 
            yanchor="bottom",
            layer="above"
        )
    )

## Now, highlight the best pokemon and the last in the 10 
first_pokemon = favorites_df_sorted.iloc[0] 
last_pokemon = favorites_df_sorted.iloc[-1] 
init_xaxis = POKE_df['base_stat_total'].min() - 28

for poke in [first_pokemon, last_pokemon]:
    
    fig.add_shape(
        type="line",
        x0=init_xaxis, y0=poke['votes'],            
        x1=poke['base_stat_total'], y1=poke['votes'],  
        xref="x", yref="y",                            
        line=dict(color="#B1AFB5", width=1.5, dash="dot"),     
        layer="above"
    )
    
    fig.add_annotation(
        x=init_xaxis-47, 
        y=poke['votes'],
        xref="x", yref="y",
        text=f"- <b>{poke['votes']:,}</b>", 
        showarrow=False,
        xanchor="left", yanchor="middle",
        font=dict(family=poke_font, size=16, color="#5A5665"),
        borderwidth=2,
        borderpad=5
    )

## ................... ##
## BOREDERS and legend ##
## ................... ##

## Plot border
fig.add_shape(
    type="rect",
    xref="paper", yref="paper", 
    x0=0, y0=0, x1=1, y1=1,     
    line=dict(color="#5A5665", width=5),
    layer="above"
)

## Background box for the legend on right side
fig.add_shape(
    type="rect",
    xref="paper", yref="paper",
    x0=1.02, y0=0.15, x1=1.24, y1=1.0,  # X and Y boundaries of the box
    fillcolor="white",
    line=dict(color="#5A5665", width=5), # Thick retro border
    layer="below"

)

## Title of the plot text annotation
fig.add_annotation(
    xref="paper", yref="paper",
    x=1.035, y=0.94,
    text="<b>2020 Fan Favourites</b>",
    showarrow=False,
    xanchor="left", yanchor="middle",
    font=dict(family=poke_font, size=22, color="#5A5665")
)

## Adding sprites to the plot
for i, (_, row) in enumerate(favorites_df_sorted.iterrows()):
    
    ## Calculate vertical position (Starts below the title, moves down)
    y_pos = 0.88 - (i * 0.07) 

    ## Add the Sprite
    sprite_path = SPRITE_FOLDER + f"{row['name_lower']}.png"
    if os.path.exists(sprite_path):
        fig.add_layout_image(
            dict(
                source=Image.open(sprite_path),
                xref="paper", yref="paper",
                x=1.02, y=y_pos,          
                sizex=0.11, sizey=0.11,   
                xanchor="left", yanchor="middle",
                layer="above"
            )
        )
    
    ## Add the text annotation on the side of the sprite
    fig.add_annotation(
        xref="paper", yref="paper",
        x=1.06, y=y_pos-0.027,                  
        text=f" \t (#{i+1})  {row['name_lower'].upper()}",
        showarrow=False,
        xanchor="left", yanchor="middle",
        font=dict(family=poke_font, size=16, color="#5A5665")
    )


## ............. ##
## CAPTION FRAME ##
## ............. ##

## TEXT of the caption
fig.add_annotation(
    xref="paper", yref="paper",
    x=0.025, y=-0.275,
    text=f"<b>FIGURE</b>.   Base  Stats  of  all  {POKE_df.shape[0]}  POKéMONs  up  to  gen  8.  Performed  denstity  estimation  via  KDE,  where  overimposed  POKéMON  sprites resemble  in  heights  the",
    showarrow=False,
    xanchor="left", yanchor="bottom", 
    align="left",   
    font=dict(family=poke_font, size=19.5, color="#5A5665")
)

fig.add_annotation(
    xref="paper", yref="paper",
    x=0.085, y=-0.265,
    text=" 2020 POKéMON  of  the  year user  preferences.  Here  showed  the  10 highest  preferrances.  Source -  https://pokemon2020.pokemon.com/en-us/  ",
    showarrow=False,
    xanchor="left", yanchor="top",    
    align="left",                     
    font=dict(family=poke_font, size=19.5, color="#5A5665")
)

## BORDER in POKéMON style
def rounded_rect(x0, y0, x1, y1, rx, ry):
    return (
        f"M {x0+rx},{y0} "
        f"L {x1-rx},{y0} "
        f"Q {x1},{y0} {x1},{y0+ry} "
        f"L {x1},{y1-ry} "
        f"Q {x1},{y1} {x1-rx},{y1} "
        f"L {x0+rx},{y1} "
        f"Q {x0},{y1} {x0},{y1-ry} "
        f"L {x0},{y0+ry} "
        f"Q {x0},{y0} {x0+rx},{y0} Z"
    )

## Borde params
cap_x0, cap_x1 = 0.0, 1.24  
cap_y0, cap_y1 = -0.34, -0.20
dx_black, dy_black = 0.002, 0.004
dx_red, dy_red = 0.015, 0.008
rx_out = 0.015
ry_out = 0.035

## Build the actual shape with three layers
fig.add_shape(
    type="path", xref="paper", yref="paper",
    path=rounded_rect(cap_x0, cap_y0, cap_x1, cap_y1, rx_out, ry_out),
    fillcolor="#000000", line_width=0, layer="below"
)

fig.add_shape(
    type="path", xref="paper", yref="paper",
    path=rounded_rect(
        cap_x0 + dx_black, cap_y0 + dy_black, 
        cap_x1 - dx_black, cap_y1 - dy_black, 
        rx_out * 0.9, ry_out * 0.9
    ),
    fillcolor="#D64848", line_width=0, layer="below"
)

fig.add_shape(
    type="path", xref="paper", yref="paper",
    path=rounded_rect(
        cap_x0 + dx_red, cap_y0 + dy_red, 
        cap_x1 - dx_red, cap_y1 - dy_red, 
        rx_out * 0.6, ry_out * 0.6
    ),
    fillcolor="#F8F8F8", line_width=0, layer="below"
)

## ............ ##
## Saving infos ##
## ............ ##
config_download = {
  'toImageButtonOptions': {
    'format': 'png', 
    'filename': 'Vis_1_highres',
    'height': 820,       
    'scale': 4           
  }
}

fig.show(config=config_download)